In [ ]:
%sql
/* ===================================================================================
   VIEW: vw_cost_center_mapping_bootstrap
   SOURCE: hive_metastore.ra_adido_2025.fy25_cost_center_mapping
   
   BUSINESS RULES & LOGIC APPLIED:
   
   0. SMART PADDING (New Rule):
      - If CostCenterID is purely numeric AND less than 4 digits, pads leading zeros 
        to make it exactly 4 digits (e.g., '56' -> '0056').
      - If CostCenterID is 4+ digits (e.g., '2830') or alphanumeric, it remains untouched.

   1. MUTUALLY EXCLUSIVE BRANCHING (The "Yes" Rule):
      - If Additional AU column is 'Yes' (or blank): Use ONLY Primary Columns.
      - If Additional AU column is NOT 'Yes': Use ONLY Additional Columns.
      
   2. DIRTY STRING PARSING (Mashed Concatenations):
      - Uses Regex to slice the string into individual blocks strictly at every 6-digit boundary.
      
   3. HYPHEN PROTECTION & FALLBACK:
      - Extracts the 6-digit AU_ID.
      - For the remaining text: If hyphens exist, splits Name and Segment based on the 
        LAST hyphen. If no hyphens exist, treats the entire text as the AU Name.
      
   4. DATA CLEANSING:
      - Strips rogue leading/trailing double quotes.
      - Converts double/multiple spaces into a single space.
      - Strictly drops any ghost rows where the AU_ID or AU_Name is missing/blank.
      
   5. SINGLE SOURCE OF TRUTH (Deduplication & Standardization):
      - Resolves naming conflicts across different Cost Centers by using a Window 
        Function to force a single, standardized AU Name and Segment per AU_ID.
=================================================================================== */

CREATE OR REPLACE TEMP VIEW vw_cost_center_mapping_bootstrap AS

WITH Base_Data AS (
    SELECT 
        -- SMART PADDING LOGIC
        CASE 
            WHEN TRIM(CAST(`CostCenterId` AS STRING)) RLIKE '^[0-9]+$' 
             AND LENGTH(TRIM(CAST(`CostCenterId` AS STRING))) < 4 
            THEN LPAD(TRIM(CAST(`CostCenterId` AS STRING)), 4, '0')
            ELSE TRIM(CAST(`CostCenterId` AS STRING)) 
        END AS Cost_Center_ID,
        
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Primary_AU_Name,
        TRIM(`Segment`) AS Primary_Segment,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Col_E,
        TRIM(`AdditionalAUID`) AS Col_F
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Branch_Primary AS (
    -- LOGIC 1a: If Col E is 'Yes' (or blank), STRICTLY use B, C, D
    SELECT 
        Cost_Center_ID,
        Primary_AU_ID AS AU_ID,
        Primary_AU_Name AS AU_Name,
        Primary_Segment AS Segment_Name
    FROM Base_Data
    WHERE Col_E = 'Yes' 
       OR Col_E IS NULL 
       OR Col_E = ''
),

Branch_Additional_Raw AS (
    -- LOGIC 1b: If Col E is NOT 'Yes', STRICTLY use E and F (Ignore B, C, D)
    SELECT 
        Cost_Center_ID,
        CONCAT_WS(' ', COALESCE(Col_E, ''), COALESCE(Col_F, '')) AS Mashed_String
    FROM Base_Data
    WHERE Col_E != 'Yes' 
      AND Col_E IS NOT NULL 
      AND Col_E != ''
),

Extracted_Blocks AS (
    -- LOGIC 2: Slice the mashed E & F text into blocks at every 6-digit boundary
    SELECT 
        Cost_Center_ID,
        EXPLODE(regexp_extract_all(Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    FROM Branch_Additional_Raw
    WHERE Mashed_String != ''
),

Separated_ID_And_Remainder AS (
    -- LOGIC 3a: Pull out the 6-digit ID, and isolate the rest of the text
    SELECT 
        Cost_Center_ID,
        TRIM(regexp_extract(Raw_Block, '^([0-9]{6})', 1)) AS AU_ID,
        TRIM(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '')) AS Remainder
    FROM Extracted_Blocks
    WHERE TRIM(Raw_Block) != ''
),

Parsed_Additionals AS (
    -- LOGIC 3b: Smartly parse the remainder based on whether hyphens exist
    SELECT 
        Cost_Center_ID,
        AU_ID,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '^(.*)[ \t]*-[ \t]*[^-]+$', 1))
            ELSE Remainder 
        END AS AU_Name,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '.*[ \t]*-[ \t]*([^-]+)$', 1))
            ELSE '' 
        END AS Segment_Name
    FROM Separated_ID_And_Remainder
),

Cleaned_Stack AS (
    -- LOGIC 4: Combine BOTH branches, clean quotes, remove weird spaces, drop blanks
    SELECT DISTINCT 
        Cost_Center_ID, 
        AU_ID, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS AU_Name, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Segment_Name, '^"|"$', ''), '[ ]+', ' ')) AS Segment_Name 
    FROM (
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Branch_Primary
        UNION
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Parsed_Additionals
    )
    WHERE AU_ID IS NOT NULL AND AU_ID != ''
      AND AU_Name IS NOT NULL AND TRIM(AU_Name) != ''
)

-- LOGIC 5: Final Standardization (One Name/Segment per AU_ID)
SELECT DISTINCT 
    Cost_Center_ID,
    AU_ID,
    FIRST_VALUE(AU_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS AU_Name,
    FIRST_VALUE(Segment_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS Segment_Name
FROM Cleaned_Stack;

In [ ]:
%sql
WITH Base_Data AS (
    SELECT 
        -- SMART PADDING LOGIC
        CASE 
            WHEN TRIM(CAST(`CostCenterId` AS STRING)) RLIKE '^[0-9]+$' 
             AND LENGTH(TRIM(CAST(`CostCenterId` AS STRING))) < 4 
            THEN LPAD(TRIM(CAST(`CostCenterId` AS STRING)), 4, '0')
            ELSE TRIM(CAST(`CostCenterId` AS STRING)) 
        END AS Cost_Center_ID,
        
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Primary_AU_Name,
        TRIM(`Segment`) AS Primary_Segment,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Col_E,
        TRIM(`AdditionalAUID`) AS Col_F
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Branch_Primary AS (
    SELECT 
        Cost_Center_ID,
        'PRIMARY BRANCH' AS Routing_Decision,
        Col_E AS Raw_Additional_String,
        'N/A - Direct Pull' AS String_Fed_To_Regex,
        Primary_AU_ID AS AU_ID,
        Primary_AU_Name AS AU_Name,
        Primary_Segment AS Segment_Name,
        '✅ KEPT: Valid Primary AU' AS Pipeline_Status
    FROM Base_Data
    WHERE Col_E = 'Yes' 
       OR Col_E IS NULL 
       OR Col_E = ''
),

Branch_Additional_Raw AS (
    SELECT 
        Cost_Center_ID,
        Col_E AS Raw_Additional_String,
        CONCAT_WS(' ', COALESCE(Col_E, ''), COALESCE(Col_F, '')) AS Mashed_String
    FROM Base_Data
    WHERE Col_E != 'Yes' 
      AND Col_E IS NOT NULL 
      AND Col_E != ''
),

Extracted_Blocks AS (
    SELECT 
        Cost_Center_ID,
        'ADDITIONAL BRANCH' AS Routing_Decision,
        Raw_Additional_String,
        Mashed_String AS String_Fed_To_Regex,
        EXPLODE(regexp_extract_all(Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    FROM Branch_Additional_Raw
    WHERE Mashed_String != ''
),

Parsed_Additionals AS (
    SELECT 
        Cost_Center_ID,
        Routing_Decision,
        Raw_Additional_String,
        String_Fed_To_Regex,
        TRIM(regexp_extract(Raw_Block, '^([0-9]{6})', 1)) AS AU_ID,
        
        CASE 
            WHEN REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '') LIKE '%-%' 
            THEN TRIM(regexp_extract(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', ''), '^(.*)[ \t]*-[ \t]*[^-]+$', 1))
            ELSE REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '') 
        END AS AU_Name,
        
        CASE 
            WHEN REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '') LIKE '%-%' 
            THEN TRIM(regexp_extract(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', ''), '.*[ \t]*-[ \t]*([^-]+)$', 1))
            ELSE '' 
        END AS Segment_Name
    FROM Extracted_Blocks
    WHERE TRIM(Raw_Block) != ''
),

Cleaned_Additionals AS (
    SELECT 
        Cost_Center_ID,
        Routing_Decision,
        Raw_Additional_String,
        String_Fed_To_Regex,
        AU_ID,
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS AU_Name,
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Segment_Name, '^"|"$', ''), '[ ]+', ' ')) AS Segment_Name,
        CASE 
            WHEN AU_ID IS NULL OR AU_ID = '' THEN '❌ DROPPED: No valid 6-digit ID found'
            WHEN AU_Name IS NULL OR TRIM(AU_Name) = '' THEN '❌ DROPPED: ID found, but Name is blank'
            ELSE '✅ KEPT: Regex successfully parsed ID and Name'
        END AS Pipeline_Status
    FROM Parsed_Additionals
),

Combined_Sim AS (
    SELECT Cost_Center_ID, Routing_Decision, Raw_Additional_String, String_Fed_To_Regex, AU_ID, AU_Name, Segment_Name, Pipeline_Status 
    FROM Branch_Primary
    
    UNION ALL
    
    SELECT Cost_Center_ID, Routing_Decision, Raw_Additional_String, String_Fed_To_Regex, AU_ID, AU_Name, Segment_Name, Pipeline_Status 
    FROM Cleaned_Additionals
)

-- To test a specific Cost Center, uncomment the WHERE clause below:
SELECT * FROM Combined_Sim
-- WHERE Cost_Center_ID IN ('2830', '2831', '0056')
ORDER BY Cost_Center_ID;